# Financial RAG

Retrieval-Augmented Generation combines document retrieval with answer generation. This notebook builds an offline financial RAG prototype using simple TF-IDF retrieval.

Abbreviations used in this notebook:

- **RAG**: Retrieval-Augmented Generation.
- **TF-IDF**: Term Frequency-Inverse Document Frequency.
- **TF**: Term Frequency, how often a term appears in a document.
- **IDF**: Inverse Document Frequency, how rare a term is across documents.
- **DCF**: Discounted Cash Flow.
- **FCF**: Free Cash Flow.
- **WACC**: Weighted Average Cost of Capital.
- **LLM**: Large Language Model.

## 1. Intuition

A financial RAG system answers questions by first finding relevant source material, then using those sources to support a response. The retrieval step is crucial because it controls what evidence the system sees.

In production, an LLM would synthesize the final answer. Here we keep everything deterministic and offline so the mechanics are visible.

## 2. Mathematics

TF-IDF weight:

$$
Weight_{term,doc} = TF_{term,doc} \times IDF_{term}
$$

Where:

- $Weight_{term,doc}$ = TF-IDF weight for a term in a document
- $TF_{term,doc}$ = term frequency inside a document
- $IDF_{term}$ = inverse document frequency across the corpus

Cosine similarity:

$$
Similarity(q,d) = \frac{q \cdot d}{||q||\ ||d||}
$$

Where:

- $Similarity(q,d)$ = cosine similarity between question and document vectors
- $q$ = vector representation of the question
- $d$ = vector representation of a document

RAG flow:

$$
Question \rightarrow Retrieval \rightarrow Context \rightarrow Answer
$$

Where:

- $Question$ = user question or analytical prompt
- $Retrieval$ = process of finding relevant source documents
- $Context$ = retrieved information passed into the answer step
- $Answer$ = generated response based on the retrieved context

## 3. Implementation

We create a synthetic document set, retrieve relevant documents for a question, and produce a grounded answer template.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "06_ai_agents" / "ai_utils.py"
spec = importlib.util.spec_from_file_location("ai_utils", helper_path)
ai_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ai_utils)

plt.style.use("seaborn-v0_8-whitegrid")
documents = ai_utils.sample_financial_documents()

documents

In [ ]:
question = "Is Nestle free cash flow resilient and what risks matter for valuation?"
retrieved = ai_utils.retrieve(documents, question, top_k=4, ticker="NESN.SW")
retrieved[["doc_id", "source", "section", "score", "text"]]

In [ ]:
def grounded_answer(question, retrieved_documents):
    bullets = []
    for row in retrieved_documents.itertuples(index=False):
        bullets.append(f"- {row.section}: {row.text}")
    return "Question: " + question + "\n\nRetrieved evidence:\n" + "\n".join(bullets)

print(grounded_answer(question, retrieved))

## 4. Visualization

Retrieval scores help diagnose whether the system found useful evidence or weakly related text.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
retrieved.sort_values("score").plot(x="doc_id", y="score", kind="barh", ax=ax, color="#2f6f8f", legend=False)
ax.set_title("Retrieved Document Scores")
ax.set_xlabel("Cosine similarity")
plt.tight_layout(); plt.show()

## 5. Application

Financial RAG can support analyst workflows: retrieving financial statement notes, risk factors, valuation assumptions, market data explanations, and prior investment theses.

The important rule is traceability. Every generated claim should be linked back to retrieved evidence.

In [ ]:
coverage = retrieved.groupby(["source", "section"]).size().to_frame("retrieved_count")
coverage

## 6. Reflection

- RAG quality depends on retrieval quality.
- Metadata filters reduce irrelevant context.
- Source traceability matters in financial analysis.
- Offline prototypes are useful before connecting an LLM or vector database.

Questions to answer after running the notebook:

1. Which document was most relevant?
2. Did retrieval cover both cash flow and risk?
3. What metadata would improve retrieval?
4. What could go wrong if retrieval misses an important risk?